# MCP Server

Starts the FastMCP server that provides agent card discovery (via embeddings), travel data queries, and Google Places lookup.

In [1]:
import json
import logging
import os
import sqlite3
import sys
from pathlib import Path

import numpy as np
import requests
import weave
from openai import OpenAI
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger(__name__)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

AGENT_CARDS_DIR = 'agent_cards'
EMBEDDING_MODEL = 'text-embedding-3-small'
SQLITE_DB = 'travel_agency.db'
PLACES_API_URL = 'https://places.googleapis.com/v1/places:searchText'
HOST, PORT = 'localhost', 10100

oai = OpenAI()

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


2026-04-16 10:05:05,820 INFO httpx: HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
2026-04-16 10:05:06,135 INFO httpx: HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
2026-04-16 10:05:06,471 INFO httpx: HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
2026-04-16 10:05:06,543 INFO httpx: HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
2026-04-16 10:05:06,663 INFO httpx: HTTP Request: GET https://pypi.org/pypi/wandb/json "HTTP/1.1 200 OK"
2026-04-16 10:05:06,821 INFO httpx: HTTP Request: GET https://pypi.org/pypi/weave/json "HTTP/1.1 200 OK"


weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


2026-04-16 10:05:06,838 INFO weave.trace.init_message: Logged in as Weights & Biases user: paulbruffett.
View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave
Weave initialized: pbruffett/a2a-lab


In [2]:
def load_agent_cards():
    card_uris, agent_cards = [], []
    for filename in sorted(os.listdir(AGENT_CARDS_DIR)):
        if filename.endswith('.json'):
            with open(Path(AGENT_CARDS_DIR) / filename) as f:
                data = json.load(f)
            card_uris.append(f'resource://agent_cards/{Path(filename).stem}')
            agent_cards.append(data)
    return card_uris, agent_cards


def build_agent_card_embeddings():
    card_uris, agent_cards = load_agent_cards()
    texts = [json.dumps(card) for card in agent_cards]
    result = oai.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    embeddings = [item.embedding for item in result.data]
    print(f'Loaded {len(card_uris)} agent cards with embeddings')
    return card_uris, agent_cards, np.array(embeddings)


card_uris, agent_cards, card_embeddings = build_agent_card_embeddings()

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9740-ea2f-7c37-982e-33a4da90ddb5


2026-04-16 10:05:07,125 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9740-ea2f-7c37-982e-33a4da90ddb5
2026-04-16 10:05:08,416 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
2026-04-16 10:05:08,928 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Loaded 5 agent cards with embeddings


In [3]:
mcp = FastMCP('agent-cards', host=HOST, port=PORT)


@mcp.tool(name='find_agent', description='Find the most relevant agent card for a query.')
@weave.op()
def find_agent(query: str) -> str:
    logger.info('[mcp.find_agent] query=%r', query)
    result = oai.embeddings.create(input=[query], model=EMBEDDING_MODEL)
    query_embedding = result.data[0].embedding
    scores = card_embeddings @ query_embedding
    chosen = agent_cards[int(np.argmax(scores))]
    logger.info('[mcp.find_agent] chosen=%s', chosen.get('name'))
    return json.dumps(chosen)


@mcp.tool()
@weave.op()
def query_places_data(query: str):
    """Query Google Places."""
    logger.info('[mcp.query_places_data] query=%r', query)
    api_key = os.getenv('GOOGLE_PLACES_API_KEY')
    if not api_key:
        return {'places': []}
    headers = {
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': 'places.id,places.displayName,places.formattedAddress',
        'Content-Type': 'application/json',
    }
    try:
        resp = requests.post(PLACES_API_URL, headers=headers, json={'textQuery': query, 'languageCode': 'en', 'maxResultCount': 10})
        resp.raise_for_status()
        data = resp.json()
        logger.info('[mcp.query_places_data] places=%d', len(data.get('places', [])))
        return data
    except Exception as e:
        logger.warning('[mcp.query_places_data] error=%s', e)
        return {'places': []}


@mcp.tool()
@weave.op()
def query_travel_data(query: str) -> dict:
    """Query the travel SQLite database (flights, hotels, rental_cars). Only SELECT queries."""
    logger.info('[mcp.query_travel_data] query=%r', query)
    if not query or not query.strip().upper().startswith('SELECT'):
        raise ValueError(f'Invalid query: {query}')
    try:
        with sqlite3.connect(SQLITE_DB) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.cursor().execute(query).fetchall()
            result = json.dumps({'results': [dict(row) for row in rows]})
            logger.info('[mcp.query_travel_data] rows=%d', len(rows))
            return result
    except Exception as e:
        logger.warning('[mcp.query_travel_data] error=%s', e)
        return {'error': str(e)}


@mcp.resource('resource://agent_cards/list', mime_type='application/json')
def get_agent_cards() -> dict:
    return {'agent_cards': list(card_uris)}


@mcp.resource('resource://agent_cards/{card_name}', mime_type='application/json')
def get_agent_card(card_name: str) -> dict:
    uri = f'resource://agent_cards/{card_name}'
    matches = [card for u, card in zip(card_uris, agent_cards) if u == uri]
    return {'agent_card': matches}

In [ ]:
print(f'MCP Server running at http://{HOST}:{PORT}')
await mcp.run_sse_async()

MCP Server running at http://localhost:10100


INFO:     Started server process [8284]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10100 (Press CTRL+C to quit)


2026-04-16 10:05:10,517 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:56682 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:56683 - "POST /messages/?session_id=dc949bea6d2d4cec9e77dcf8b2627058 HTTP/1.1" 202 Accepted
INFO:     ::1:56683 - "POST /messages/?session_id=dc949bea6d2d4cec9e77dcf8b2627058 HTTP/1.1" 202 Accepted
INFO:     ::1:56683 - "POST /messages/?session_id=dc949bea6d2d4cec9e77dcf8b2627058 HTTP/1.1" 202 Accepted
2026-04-16 10:05:22,309 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9741-2586-7cb8-ad42-533f47e567b9


2026-04-16 10:05:22,314 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9741-2586-7cb8-ad42-533f47e567b9
2026-04-16 10:05:23,656 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:56691 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:56692 - "POST /messages/?session_id=a2e9ef69d3484a18a472a1ceefc887da HTTP/1.1" 202 Accepted
INFO:     ::1:56692 - "POST /messages/?session_id=a2e9ef69d3484a18a472a1ceefc887da HTTP/1.1" 202 Accepted
INFO:     ::1:56692 - "POST /messages/?session_id=a2e9ef69d3484a18a472a1ceefc887da HTTP/1.1" 202 Accepted
2026-04-16 10:05:40,111 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


...

2026-04-16 10:13:08,605 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-42fc-7e87-a409-3377b71272a9
2026-04-16 10:13:08,828 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-16 10:13:08,838 INFO __main__: [mcp.find_agent] chosen=Hotel Booking Agent
INFO:     ::1:56821 - "POST /messages/?session_id=b74cd457717c46b8a7c71321ac95d297 HTTP/1.1" 202 Accepted
2026-04-16 10:13:08,847 INFO mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-04-16 10:13:09,909 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:56824 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:56825 - "POST /messages/?session_id=2bbe9a8e09144b33a9f36bad52bfbc91 HTTP/1.1" 202 Accepted
INFO:     ::1:56825 - "POST /messages/?session_id=2bbe9a8e09144b33a9f36bad52bfbc91 HTTP/1.1" 202 Accepted
INFO:     ::1:56825 - "POST /messages/?session_id=2bbe9a8e09144b33a9f36bad52bfbc91 HTTP/1

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-5005-7738-b367-2244e9036d67


2026-04-16 10:13:11,942 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-5005-7738-b367-2244e9036d67
2026-04-16 10:13:12,164 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-16 10:13:12,175 INFO __main__: [mcp.find_agent] chosen=Hotel Booking Agent
INFO:     ::1:56825 - "POST /messages/?session_id=2bbe9a8e09144b33a9f36bad52bfbc91 HTTP/1.1" 202 Accepted
2026-04-16 10:13:12,184 INFO mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-04-16 10:13:13,508 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:56827 - "POST /messages/?session_id=b8b62af1a2b143e48162a38679a4ce3a HTTP/1.1" 202 Accepted
2026-04-16 10:13:13,705 INFO mcp.server.lowlevel.server: Processing request of type CallToolRequest
2026-04-16 10:13:13,706 INFO __main__: [mcp.query_travel_data] query="SELECT name, city, hotel_type, room_type, price_per_night FROM hotels WHER

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-56ea-7149-84f0-58f71c085629


2026-04-16 10:13:13,707 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-56ea-7149-84f0-58f71c085629
2026-04-16 10:13:13,708 INFO __main__: [mcp.query_travel_data] rows=1
2026-04-16 10:13:15,004 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
